# Reproducing: Keplerian Eigenvalue Structure and Spectral Rigidity in the 3-Body Coupling Tensor

**Andrew H. Bond**, San Jose State University

This notebook reproduces the key results from the paper. No GPU required for Theorems 1-4; GPU needed only for the orbit integration (Conjecture 1).

```
pip install numpy scipy tensorly matplotlib
```

In [ ]:
import numpy as np
import sys, os
sys.path.insert(0, '..')

from tensor_3body.hamiltonian import hessian_analytical, jacobi_masses
from tensor_3body.tensor_ops import reshape_to_rank6, effective_rank, singular_values
from tensor_3body.sampling import config_to_phase_space_circular
print('Imports OK')

## Theorem 1: Phase Separability

**Claim:** The qp cross-block of the Hessian is identically zero at every configuration.

**Test:** Compute at 1000 random configs, verify max|qp| = 0.

In [ ]:
rng = np.random.RandomState(42)
max_qp = 0.0
for _ in range(1000):
    r1, r2 = 10**rng.uniform(-1, 2, 2)
    theta, phi = rng.uniform(0, np.pi), rng.uniform(0, 2*np.pi)
    z = config_to_phase_space_circular(r1, r2, theta, phi, 1, 1, 1)
    H = hessian_analytical(z, 1, 1, 1)
    qp_norm = np.linalg.norm(H[0:6, 6:12])
    max_qp = max(max_qp, qp_norm)

print(f'Theorem 1: max |qp block| across 1000 configs = {max_qp:.2e}')
assert max_qp < 1e-14, 'FAILED'
print('CONFIRMED: qp block is identically zero')

## Theorem 2: Momentum Block Rigidity

**Claim:** pp eigenvalues are exactly {1/μ₁, 1/μ₁, 1/μ₁, 1/μ₂, 1/μ₂, 1/μ₂} at every configuration.

In [ ]:
mu1, mu2 = jacobi_masses(1.0, 1.0, 1.0)
expected_pp = np.sort([1/mu1]*3 + [1/mu2]*3)
print(f'Expected pp eigenvalues: {expected_pp}')

max_error = 0.0
for _ in range(1000):
    r1, r2 = 10**rng.uniform(-1, 2, 2)
    theta, phi = rng.uniform(0, np.pi), rng.uniform(0, 2*np.pi)
    z = config_to_phase_space_circular(r1, r2, theta, phi, 1, 1, 1)
    H = hessian_analytical(z, 1, 1, 1)
    pp_eigs = np.sort(np.linalg.eigvalsh(H[6:12, 6:12]))
    error = np.max(np.abs(pp_eigs - expected_pp))
    max_error = max(max_error, error)

print(f'Theorem 2: max eigenvalue error across 1000 configs = {max_error:.2e}')
assert max_error < 1e-12, 'FAILED'
print('CONFIRMED: pp eigenvalues are configuration-independent')

## Theorem 3: Keplerian Eigenvalue Structure at Rank 9

**Claim:** At rank-9 configs, V_qq eigenvalues are in exact 2:1:1:0:0:0 ratio.

In [ ]:
# Find rank-9 configurations: small r1, large r2 (tight binary + distant third body)
rank9_configs = []
for _ in range(10000):
    r1 = 10**rng.uniform(-1, 0.5)   # 0.1 to ~3
    r2 = 10**rng.uniform(1, 2)      # 10 to 100
    theta, phi = rng.uniform(0, np.pi), rng.uniform(0, 2*np.pi)
    z = config_to_phase_space_circular(r1, r2, theta, phi, 1, 1, 1)
    H = hessian_analytical(z, 1, 1, 1)
    rank = effective_rank(H)
    if rank == 9:
        qq_eigs = np.sort(np.linalg.eigvalsh(H[0:6, 0:6]))[::-1]
        rank9_configs.append({'r1': r1, 'r2': r2, 'qq_eigs': qq_eigs})

print(f'Found {len(rank9_configs)} rank-9 configurations')

# Check the 2:1 ratio
max_ratio_error = 0.0
for cfg in rank9_configs[:100]:
    e = cfg['qq_eigs']
    nonzero = np.abs(e[np.abs(e) > 1e-6 * np.abs(e).max()])
    if len(nonzero) >= 2:
        sorted_nz = np.sort(nonzero)[::-1]
        ratio = sorted_nz[0] / sorted_nz[1]
        ratio_error = abs(ratio - 2.0)
        max_ratio_error = max(max_ratio_error, ratio_error)

print(f'Theorem 3: max |ratio - 2.0| = {max_ratio_error:.2e}')
print(f'Example qq eigenvalues: {rank9_configs[0]["qq_eigs"]}')
assert max_ratio_error < 1e-4, f'FAILED: ratio error = {max_ratio_error}'
print('CONFIRMED: Keplerian 2:1 eigenvalue ratio at rank 9')

## Theorem 4: Spectral Rigidity via Davis-Kahan

**Claim:** Rank-9 configs have ~98% spectrum-preserving perturbations; rank 7-8 have ~2%.

In [ ]:
def spectral_rigidity(H, n_pert=100, epsilon=0.01, tol=0.01):
    """Fraction of random perturbations that preserve the eigenvalue spectrum."""
    eigs0 = np.sort(np.linalg.eigvalsh(H))
    norm0 = np.linalg.norm(eigs0)
    n_preserved = 0
    for _ in range(n_pert):
        dH = rng.randn(12, 12) * epsilon
        dH = (dH + dH.T) / 2
        eigs_pert = np.sort(np.linalg.eigvalsh(H + dH))
        if np.linalg.norm(eigs_pert - eigs0) / (norm0 + 1e-30) < tol:
            n_preserved += 1
    return n_preserved / n_pert

# Sample configs at different ranks
results = {}
for target_rank, r1_range, r2_range in [
    (7, (80, 100), (80, 100)),    # widely separated
    (9, (0.1, 0.5), (20, 80)),    # tight binary + distant
    (12, (0.5, 5), (0.5, 5)),     # comparable separations
]:
    rigidities = []
    gaps = []
    for _ in range(10):
        r1 = rng.uniform(*r1_range)
        r2 = rng.uniform(*r2_range)
        theta, phi = rng.uniform(0, np.pi), rng.uniform(0, 2*np.pi)
        z = config_to_phase_space_circular(r1, r2, theta, phi, 1, 1, 1)
        H = hessian_analytical(z, 1, 1, 1)
        rank = effective_rank(H)
        if rank == target_rank:
            rig = spectral_rigidity(H)
            eigs = np.sort(np.linalg.eigvalsh(H))
            gap = np.max(np.diff(eigs))
            rigidities.append(rig)
            gaps.append(gap)
    if rigidities:
        results[target_rank] = (np.mean(rigidities), np.mean(gaps))
        print(f'Rank {target_rank}: rigidity={np.mean(rigidities):.2f}, '
              f'max_gap={np.mean(gaps):.1f} (n={len(rigidities)})')

if 9 in results and 12 in results:
    print(f'\nTheorem 4: Rank-9 rigidity ({results[9][0]:.2f}) vs '
          f'rank-12 ({results[12][0]:.2f})')
    print(f'Gap ratio: {results[9][1]:.0f} / {results[12][1]:.0f} = '
          f'{results[9][1]/results[12][1]:.0f}x')
    print('Davis-Kahan predicts: larger gap -> higher rigidity')
    if results[9][0] > results[12][0]:
        print('CONFIRMED')
    else:
        print('NOT CONFIRMED at this sample size')

## Rank Landscape (Phase 1)

Quick version: 57,600 configs (coarse grid). Full version: 3.24M configs.

In [ ]:
from tensor_3body.sampling import make_coarse_grid
from tensor_3body.landscape import compute_landscape

configs = make_coarse_grid(n_r=20, n_angle=12)
print(f'Computing rank at {len(configs)} configurations...')
data = compute_landscape(configs, m1=1.0, m2=1.0, m3=1.0, n_workers=4)

ranks = data['eff_rank']
for r in range(7, 13):
    count = (ranks == r).sum()
    if count > 0:
        print(f'  rank={r}: {count} ({100*count/len(ranks):.2f}%)')

## ML Predictor (requires scikit-learn)

Load the pre-computed 51K labeled dataset and train the decision tree.

In [ ]:
# Load pre-computed results (or skip if not available)
try:
    d = np.load('../zenodo_data/prediction_large.npz', allow_pickle=True)
    from sklearn.tree import DecisionTreeClassifier, export_text
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    from sklearn.ensemble import GradientBoostingClassifier

    valid = ~d['collision']
    X = np.column_stack([
        d['ranks'][valid],
        d['inner_ranks'][valid],
        d['gammas'][valid],
        np.log10(d['freq_ratios'][valid] + 1),
        d['n_clusters'][valid],
        d['energies'][valid],
        np.log10(d['r2_r1'][valid] + 1),
    ])
    X = np.nan_to_num(X, nan=0, posinf=100, neginf=-100)
    y = d['periodic'][valid].astype(int)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=50,
                                 class_weight='balanced', random_state=42)
    dt_f1 = cross_val_score(dt, X, y, cv=cv, scoring='f1').mean()

    gb = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
    gb_f1 = cross_val_score(gb, X, y, cv=cv, scoring='f1').mean()

    print(f'Decision Tree F1: {dt_f1:.3f}')
    print(f'Gradient Boosting F1: {gb_f1:.3f}')

    dt.fit(X, y)
    feature_names = ['rank', 'inner_rank', 'gamma', 'log_freq_ratio',
                     'n_clusters', 'energy', 'log_r2_r1']
    print('\nDecision tree:')
    print(export_text(dt, feature_names=feature_names, max_depth=3))
except FileNotFoundError:
    print('Pre-computed dataset not found. Run run_prediction_large.py first.')
except ImportError:
    print('scikit-learn not installed. pip install scikit-learn')

## Summary

| Theorem | Statement | Status |
|---------|-----------|--------|
| 1. Phase Separability | qp block = 0 | Proven + confirmed |
| 2. Momentum Rigidity | pp eigenvalues constant | Proven + confirmed |
| 3. Keplerian Structure | 2:1:1 ratio at rank 9 | Proven + confirmed |
| 4. Spectral Rigidity | Davis-Kahan → 98% preservation | Proven + confirmed |
| Conjecture 1 | Rigidity → periodicity | Empirically supported |